# Silver Layer Orchestrator – fact_sell_in

**Purpose:**  
Enterprise-grade orchestration for Silver layer Sell-In Fact Table. Ensures contract-driven execution, quality gates, and operational observability. No business logic, only orchestration.

**Flow:**
1. **Transformation** (`run_transformation_sell_in`) – Cleans, standardizes, and loads sell-in data. Returns contract JSON.
2. **Validation** (`run_validation_sell_in`) – Applies business-driven data quality rules. Returns contract JSON.
3. **Monitoring** (`run_monitoring_sell_in`) – Interprets validation metrics, evaluates health, and detects trends. Returns contract JSON.

**Tables involved:**
- `workspace.silver.fact_sell_in` (transformation output)
- `workspace.silver.validation_fact_sell_in_results` (record-level validation)
- `workspace.silver.validation_fact_sell_in_metrics` (aggregated validation metrics)
- `workspace.silver.monitoring_fact_sell_in` (monitoring output)
- `workspace.silver.orchestration_log_fact_sell_in` (orchestration log)

**Architecture:** Medallion, Databricks Serverless, Unity Catalog

**Quality Gates:**
- ✅ Transformation must return `status='SUCCESS'` and `records_written > 0`
- ✅ Validation: `valid_percentage >= quality_threshold` (configurable)
- ⚠️ Monitoring: Logs warning if `has_critical_alert=True`

**Exit Codes:**
- `0`: SUCCESS
- `1`: TRANSFORMATION_FAILED
- `2`: VALIDATION_FAILED (quality gate)
- `3`: MONITORING_FAILED

**Design Principles:**
- No business logic in orchestrator
- No .collect(), .count(), cache(), persist()
- No mutation of Silver data outside scripts
- All thresholds hardcoded for auditability (see script docstrings)
- All outputs are Delta tables, partitioned for performance

**References:**
- See transformation_fact_sell_in.py, validate_fact_sell_in.py, monitor_fact_sell_in.py for business logic and technical details.


In [ ]:
import logging
from datetime import datetime
import uuid
import json
from pyspark.sql import SparkSession, functions as F

# Configure logging
logger = logging.getLogger("silver_sell_in_orchestrator")
if not logger.hasHandlers():
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info('=' * 80)
logger.info('SILVER LAYER ORCHESTRATOR - fact_sell_in')
logger.info('=' * 80)


In [ ]:
# Create widgets for configuration (Databricks only)
try:
    dbutils.widgets.text('quality_threshold', '95.0', 'Min Valid %')
    dbutils.widgets.dropdown('fail_on_quality_gate', 'true', ['true', 'false'], 'Fail on Quality Gate')
    QUALITY_THRESHOLD = float(dbutils.widgets.get('quality_threshold'))
    FAIL_ON_QUALITY_GATE = dbutils.widgets.get('fail_on_quality_gate').lower() == 'true'
    logger.info('Configuration loaded from widgets:')
except Exception:
    QUALITY_THRESHOLD = 95.0
    FAIL_ON_QUALITY_GATE = True
    logger.info('Configuration using defaults (no widgets available):')

logger.info(f'  - quality_threshold: {QUALITY_THRESHOLD}%')
logger.info(f'  - fail_on_quality_gate: {FAIL_ON_QUALITY_GATE}')


In [ ]:
# Get or create Spark session
spark = SparkSession.getActiveSession()
if spark is None:
    logger.info('No active SparkSession found, creating new one...')
    spark = SparkSession.builder.appName('Silver_fact_sell_in_orchestrator').getOrCreate()
else:
    logger.info('Using active SparkSession')
logger.info(f'Spark version: {spark.version}')


In [ ]:
# CELL 5: Import Orchestration Functions
# This cell ensures robust, orchestrator-ready imports for the Sell-In Silver pipeline.
# - Uses sys.path to add the relative module directory (../fact_sell_in/)
# - Forces module reload with importlib to avoid cache/shadowing issues in Databricks/Jupyter
# - Imports transformation, validation, and monitoring entrypoints with clear aliases
# - All logic is orchestrator-only: no business logic, no data mutation

import sys
import importlib
import os

sys.path.append("../fact_sell_in/")

def force_reload(module_name):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

try:
    force_reload("transformation_fact_sell_in")
    force_reload("validate_fact_sell_in")
    force_reload("monitor_fact_sell_in")
    from transformation_fact_sell_in import run_silver_fact_sell_in_transformation as run_transformation_sell_in
    from validate_fact_sell_in import run_silver_fact_sell_in_validation as run_validation_sell_in
    from monitor_fact_sell_in import SilverFactSellInMonitor
    
    logger.info("✅ Successfully imported orchestration functions (forced reload)")
except ImportError as e:
    logger.error(f"❌ Failed to import: {e}")
    raise

def run_monitoring_sell_in(spark, transformation_result, validation_result):
    monitor = SilverFactSellInMonitor()
    return monitor.monitor(spark, transformation_result, validation_result)


In [ ]:
# Initialize execution context
run_id = str(uuid.uuid4())
execution_timestamp = datetime.utcnow()
summary = {
    'run_id': run_id,
    'execution_timestamp': execution_timestamp.isoformat(),
    'transformation': {},
    'validation': {},
    'monitoring': {},
    'status': 'STARTED',
    'error_message': None,
    'exit_code': 0
}


In [ ]:
logger.info('')
logger.info('=' * 80)
logger.info('STEP 1/3: SILVER TRANSFORMATION')
logger.info('=' * 80)
try:
    transformation_result = run_transformation_sell_in(spark=spark)
    # Validate contract
    if not isinstance(transformation_result, dict):
        raise Exception('Transformation contract not a dict')
    required_fields = ['status', 'records_read', 'records_written', 'duration', 'status']
    for field in required_fields:
        if field not in transformation_result:
            raise Exception(f'Missing field in transformation contract: {field}')
    summary['transformation'] = transformation_result
    logger.info(f"Transformation Status: {transformation_result['status']}")
    logger.info(f"Records Read: {transformation_result.get('records_read', 'N/A')}")
    logger.info(f"Records Written: {transformation_result.get('records_written', 'N/A')}")
    logger.info(f"Duration: {transformation_result.get('duration', 'N/A')}s")
    if transformation_result['status'] != 'success':
        error_val = transformation_result.get('error_message', 'Unknown error')
        error_msg = f'Transformation failed: {error_val}'
        logger.error(f'❌ {error_msg}')
        summary['status'] = 'FAILED'
        summary['error_message'] = error_msg
        summary['exit_code'] = 1
        raise Exception(error_msg)
    if transformation_result.get('records_written', 0) == 0:
        error_msg = 'Transformation wrote zero records'
        logger.error(f'❌ {error_msg}')
        summary['status'] = 'FAILED'
        summary['error_message'] = error_msg
        summary['exit_code'] = 1
        raise Exception(error_msg)
    logger.info('✅ Transformation completed successfully')
except Exception as e:
    logger.error(f'❌ Transformation step failed: {str(e)}', exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 1
    raise


In [ ]:
logger.info('')
logger.info('=' * 80)
logger.info('STEP 2/3: SILVER VALIDATION')
logger.info('=' * 80)
try:
    validation_result = run_validation_sell_in(spark=spark)
    if not isinstance(validation_result, dict):
        raise Exception('Validation contract not a dict')
    required_fields = ['status', 'total_records', 'validation_results', 'failed_validations', 'duration_seconds']
    for field in required_fields:
        if field not in validation_result:
            raise Exception(f'Missing field in validation contract: {field}')
    summary['validation'] = validation_result
    # For sell_in, use a simple pass/fail based on failed_validations and status
    valid_percentage = 100.0 if validation_result.get('failed_validations', 0) == 0 else 0.0
    logger.info(f"Total Records: {validation_result.get('total_records', 'N/A')}")
    logger.info(f"Failed Validations: {validation_result.get('failed_validations', 'N/A')}")
    logger.info('🚦 QUALITY GATE CHECK')
    logger.info(f'   Threshold: {QUALITY_THRESHOLD}%')
    logger.info(f'   Actual: {valid_percentage:.2f}%')
    if validation_result['status'] != 'success':
        error_val = validation_result.get('error_message', 'Unknown error')
        error_msg = f'Validation failed: {error_val}'
        logger.error(f'❌ {error_msg}')
        summary['status'] = 'FAILED'
        summary['error_message'] = error_msg
        summary['exit_code'] = 2
        raise Exception(error_msg)
    if valid_percentage < QUALITY_THRESHOLD:
        error_msg = f'Quality gate failed: {valid_percentage:.2f}% < {QUALITY_THRESHOLD}%'
        logger.error(f'❌ {error_msg}')
        if FAIL_ON_QUALITY_GATE:
            summary['status'] = 'FAILED'
            summary['error_message'] = error_msg
            summary['exit_code'] = 2
            raise Exception(error_msg)
        else:
            logger.warning('⚠️ Quality gate failed but continuing (fail_on_quality_gate=false)')
    else:
        logger.info('✅ Quality gate passed')
    logger.info('✅ Validation completed successfully')
except Exception as e:
    logger.error(f'❌ Validation step failed: {str(e)}', exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 2
    raise


In [ ]:
logger.info('')
logger.info('=' * 80)
logger.info('STEP 3/3: SILVER MONITORING')
logger.info('=' * 80)
try:
    monitoring_result = run_monitoring_sell_in(spark=spark, transformation_result=summary['transformation'], validation_result=summary['validation'])
    if not isinstance(monitoring_result, dict):
        raise Exception('Monitoring contract not a dict')
    required_fields = ['pipeline_status', 'severity', 'event_timestamp', 'human_message']
    for field in required_fields:
        if field not in monitoring_result:
            raise Exception(f'Missing field in monitoring contract: {field}')
    summary['monitoring'] = monitoring_result
    logger.info(f"Pipeline Status: {monitoring_result.get('pipeline_status', 'N/A')}")
    logger.info(f"Severity: {monitoring_result.get('severity', 'N/A')}")
    logger.info(f"Message: {monitoring_result.get('human_message', 'N/A')}")
    if monitoring_result.get('severity', '').lower() == 'critical':
        logger.warning('⚠️' * 40)
        logger.warning(f"CRITICAL ALERT DETECTED: {monitoring_result.get('human_message', '')}")
        logger.warning('⚠️' * 40)
    logger.info('✅ Monitoring completed successfully')
except Exception as e:
    logger.error(f'❌ Monitoring step failed: {str(e)}', exc_info=True)
    if summary['exit_code'] == 0:
        summary['status'] = 'FAILED'
        summary['error_message'] = str(e)
        summary['exit_code'] = 3
    raise


In [ ]:
# Persist Orchestration Log for sell_in
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, BooleanType
)
def _safe_str(value):
    return json.dumps(value, default=str) if value is not None else None

orchestration_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp", StringType(), False),
    StructField("status", StringType(), False),
    StructField("exit_code", IntegerType(), False),
    StructField("error_message", StringType(), True),
    # Transformation
    StructField("transformation_status", StringType(), True),
    StructField("transformation_records_read", LongType(), True),
    StructField("transformation_records_written", LongType(), True),
    StructField("transformation_duration", DoubleType(), True),
    # Validation
    StructField("validation_status", StringType(), True),
    StructField("validation_total_records", LongType(), True),
    StructField("validation_failed_validations", IntegerType(), True),
    StructField("validation_duration_seconds", DoubleType(), True),
    # Monitoring
    StructField("monitoring_pipeline_status", StringType(), True),
    StructField("monitoring_severity", StringType(), True),
    StructField("monitoring_event_timestamp", StringType(), True),
    StructField("monitoring_human_message", StringType(), True),
    # Full JSON
    StructField("full_summary_json", StringType(), False),
])

orchestration_log_row = {
    "run_id": str(summary["run_id"]),
    "execution_timestamp": str(summary["execution_timestamp"]),
    "status": str(summary["status"]),
    "exit_code": int(summary["exit_code"]),
    "error_message": summary.get("error_message"),
    # Transformation
    "transformation_status": summary["transformation"].get("status"),
    "transformation_records_read": summary["transformation"].get("records_read"),
    "transformation_records_written": summary["transformation"].get("records_written"),
    "transformation_duration": summary["transformation"].get("duration"),
    # Validation
    "validation_status": summary["validation"].get("status"),
    "validation_total_records": summary["validation"].get("total_records"),
    "validation_failed_validations": summary["validation"].get("failed_validations"),
    "validation_duration_seconds": summary["validation"].get("duration_seconds"),
    # Monitoring
    "monitoring_pipeline_status": summary["monitoring"].get("pipeline_status"),
    "monitoring_severity": summary["monitoring"].get("severity"),
    "monitoring_event_timestamp": summary["monitoring"].get("event_timestamp"),
    "monitoring_human_message": summary["monitoring"].get("human_message"),
    # Full summary
    "full_summary_json": _safe_str(summary)
}

try:
    orchestration_log_df = spark.createDataFrame(
        [orchestration_log_row],
        schema=orchestration_log_schema
    )
    orchestration_log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("workspace.silver.orchestration_log_fact_sell_in")
    logger.info("✅ Orchestration log persisted successfully")
except Exception as e:
    logger.warning(f"⚠️ Failed to persist orchestration log: {e}")


In [ ]:
# For Databricks Workflows: Exit with status
try:
    exit_payload = {
        'status': summary['status'],
        'exit_code': summary['exit_code'],
        'run_id': summary['run_id'],
        'error_message': summary['error_message'],
        'failed_validations': summary['validation'].get('failed_validations'),
        'pipeline_status': summary['monitoring'].get('pipeline_status'),
        'severity': summary['monitoring'].get('severity'),
        'human_message': summary['monitoring'].get('human_message')
    }
    logger.info(f"Exiting with payload: {exit_payload}")
    dbutils.notebook.exit(json.dumps(exit_payload))
except NameError:
    logger.info('Not in Databricks environment, skipping dbutils.notebook.exit()')
    if summary['status'] == 'FAILED':
        logger.error(f"Orchestration failed with exit code {summary['exit_code']}")
        raise Exception(f"Orchestration failed: {summary['error_message']}")
    else:
        logger.info('✅ Orchestration completed successfully in Jupyter environment')
